# 🏥 Aged Care Demand Forecasting — Australian Public Sector
## Notebook 03: Exploratory Data Analysis

> **Inputs from 02_data_collection_REAL.ipynb:**
> - `seifa_2021.csv` — IRSD, IRSAD scores + remoteness (SA2)
> - `abs_population_projections.csv` — pop_65plus, pop_70plus, projections 2031/2041 (SA2)
> - `aihw_recipients_sa2.csv` — CHSP, home care, residential recipients (SA2)
>
> **Note:** No quarterly time-series data available. AIHW data is a single snapshot
> (2024–25 for CHSP, 30 June 2025 for HCP and Residential). Section 3 reflects this.

---

## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

RAW_DIR     = Path('../data/raw')
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Load datasets ─────────────────────────────────────────────────────────────
pop_df   = pd.read_csv(RAW_DIR / 'abs_population_projections.csv', dtype={'sa2_code': str})
recv_df  = pd.read_csv(RAW_DIR / 'aihw_recipients_sa2.csv',        dtype={'sa2_code': str})
seifa_df = pd.read_csv(RAW_DIR / 'seifa_2021.csv',                 dtype={'sa2_code': str})

# Standardise SA2 codes
for df in [pop_df, recv_df, seifa_df]:
    df['sa2_code'] = df['sa2_code'].str.zfill(9)

# ── Build master dataset ──────────────────────────────────────────────────────
# Base: population projections (2,454 SA2s)
# Left join AIHW recipients — all SA2s get recipient counts (0 if not matched)
# Left join SEIFA — ~100 SA2s may be missing (unpopulated/suppressed areas)
master = (
    pop_df
    .merge(recv_df[['sa2_code', 'chsp_recipients', 'home_care_recipients',
                    'residential_recipients', 'total_recipients']],
           on='sa2_code', how='left')
    .merge(seifa_df[['sa2_code', 'irsd_score', 'irsd_decile',
                     'irsad_score', 'remoteness_cat']],
           on='sa2_code', how='left')
)

# Fill missing recipient counts with 0
for col in ['chsp_recipients', 'home_care_recipients',
            'residential_recipients', 'total_recipients']:
    master[col] = master[col].fillna(0).astype(int)

# Remoteness labels
REMOTE_LABELS = {1: 'Major City', 2: 'Inner Regional',
                 3: 'Outer Regional', 4: 'Remote', 5: 'Very Remote'}
master['remoteness_label'] = master['remoteness_cat'].map(REMOTE_LABELS)

print(f'✅ Master dataset: {master.shape}')
print(f'   SA2 regions with SEIFA data: {master["irsd_score"].notna().sum():,}')
print(f'   SA2 regions with recipients: {(master["total_recipients"] > 0).sum():,}')
master.head(3)

---
## 1. National 70+ Population Projections

Using `pop_70plus` (the demand proxy defined in notebook 01) and state-level ABS Series B growth factors.

In [ ]:
# National totals at three time points
national = {
    '2024': master['pop_70plus'].sum(),
    '2031': master['pop_70plus_2031'].sum(),
    '2041': master['pop_70plus_2041'].sum(),
}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#4C72B0', '#DD8452', '#55A868']
bars = ax.bar(list(national.keys()), [v / 1e6 for v in national.values()],
              color=colors, width=0.5, edgecolor='white')

for bar, val in zip(bars, national.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val / 1e6:.2f}M', ha='center', va='bottom', fontweight='bold')

growth_2031 = (national['2031'] / national['2024'] - 1) * 100
growth_2041 = (national['2041'] / national['2024'] - 1) * 100

ax.set_title('Projected 70+ Population — All SA2 Regions (ABS Series B)', fontsize=13, fontweight='bold')
ax.set_ylabel('Population (millions)')
ax.set_ylim(0, max(national.values()) / 1e6 * 1.2)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_01_population_projection.png', dpi=150)
plt.show()
print(f'  2024→2031 growth: +{growth_2031:.1f}%')
print(f'  2024→2041 growth: +{growth_2041:.1f}%')

---
## 2. 70+ Population Share & Growth by State

In [ ]:
state_summary = master.groupby('state').agg(
    pop_70plus_2024=('pop_70plus',      'sum'),
    pop_70plus_2031=('pop_70plus_2031', 'sum'),
    total_pop=      ('total_pop',       'sum'),
    sa2_count=      ('sa2_code',        'count'),
).reset_index()

state_summary['pct_70plus_2024'] = (
    state_summary['pop_70plus_2024'] / state_summary['total_pop'] * 100
)
state_summary['growth_pct_2031'] = (
    state_summary['pop_70plus_2031'] / state_summary['pop_70plus_2024'] - 1
) * 100
state_summary = state_summary.sort_values('pct_70plus_2024', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].barh(state_summary['state'], state_summary['pct_70plus_2024'], color='#4C72B0')
axes[0].set_xlabel('% of population aged 70+')
axes[0].set_title('70+ Population Share by State (2024)', fontweight='bold')
for i, val in enumerate(state_summary['pct_70plus_2024']):
    axes[0].text(val + 0.1, i, f'{val:.1f}%', va='center', fontsize=9)

nat_avg = state_summary['growth_pct_2031'].mean()
colors_g = ['#DD8452' if g > nat_avg else '#4C72B0' for g in state_summary['growth_pct_2031']]
axes[1].barh(state_summary['state'], state_summary['growth_pct_2031'], color=colors_g)
axes[1].set_xlabel('Growth rate 2024→2031 (%)')
axes[1].set_title('70+ Population Growth Rate by State', fontweight='bold')
axes[1].axvline(nat_avg, color='red', linestyle='--', label=f'Avg: {nat_avg:.1f}%')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_02_state_comparison.png', dpi=150)
plt.show()
print(state_summary[['state', 'pop_70plus_2024', 'pct_70plus_2024', 'growth_pct_2031']].to_string(index=False))

---
## 3. Aged Care Recipients — 2024/25 Snapshot by Care Type

> **Note:** AIHW data is a point-in-time snapshot, not a time series.
> CHSP = 2024–25 annual, Home Care & Residential = 30 June 2025.

In [ ]:
# National totals by care type
care_totals = {
    'CHSP (Home Support)':      master['chsp_recipients'].sum(),
    'Home Care Packages':       master['home_care_recipients'].sum(),
    'Residential Care':         master['residential_recipients'].sum(),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart: national totals
axes[0].bar(care_totals.keys(), [v / 1e3 for v in care_totals.values()],
            color=['#55A868', '#DD8452', '#4C72B0'], edgecolor='white')
axes[0].set_ylabel("Recipients ('000s)")
axes[0].set_title('Total Recipients by Care Type — National (2024/25)', fontweight='bold')
for i, (k, v) in enumerate(care_totals.items()):
    axes[0].text(i, v / 1e3 + 5, f'{v / 1e3:.0f}K', ha='center', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# Distribution of total_recipients per SA2
axes[1].hist(master['total_recipients'][master['total_recipients'] > 0],
             bins=40, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[1].set_xlabel('Total Recipients per SA2')
axes[1].set_ylabel('Number of SA2 regions')
axes[1].set_title('Distribution of Recipients per SA2 (excl. zero)', fontweight='bold')
axes[1].axvline(master['total_recipients'].median(), color='red', linestyle='--',
                label=f'Median: {master["total_recipients"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_03_recipients_snapshot.png', dpi=150)
plt.show()

print(f'Total recipients (all care types): {master["total_recipients"].sum():,}')
print(f'SA2 regions with any recipients:   {(master["total_recipients"] > 0).sum():,}')
print(f'SA2 regions with zero recipients:  {(master["total_recipients"] == 0).sum():,}')

---
## 4. Service Utilisation Rates — Recipients per 1,000 Elderly

Supply-side features derived from recipient counts + population (as per notebook 01 design).

In [ ]:
# Utilisation rates per 1,000 pop_70plus
master['residential_per_1000'] = (
    master['residential_recipients'] / master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0)
master['home_care_per_1000'] = (
    master['home_care_recipients'] / master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0)
master['chsp_per_1000'] = (
    master['chsp_recipients'] / master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0)
master['total_per_1000'] = (
    master['total_recipients'] / master['pop_70plus'].replace(0, np.nan) * 1000
).fillna(0)

# Cap extreme outliers for visualisation (very small pop_70plus SA2s)
cap = master['residential_per_1000'].quantile(0.99)
master_vis = master[master['residential_per_1000'] <= cap].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: residential utilisation vs pop_70plus, coloured by remoteness
colors_map = {1: '#4C72B0', 2: '#55A868', 3: '#DD8452', 4: '#C44E52', 5: '#8172B3'}
c = master_vis['remoteness_cat'].map(colors_map).fillna('#999999')
sc = axes[0].scatter(master_vis['pop_70plus'], master_vis['residential_per_1000'],
                     c=c, alpha=0.6, s=40, edgecolors='white', linewidth=0.3)
axes[0].set_xlabel('Population 70+ (2024)')
axes[0].set_ylabel('Residential Recipients per 1,000 Elderly')
axes[0].set_title('Residential Utilisation vs Elderly Population', fontweight='bold')
# Manual legend
for cat, col in colors_map.items():
    axes[0].scatter([], [], c=col, label=REMOTE_LABELS[cat], s=40)
axes[0].legend(fontsize=8, title='Remoteness')

# Box: utilisation by remoteness
remote_order = ['Major City', 'Inner Regional', 'Outer Regional', 'Remote', 'Very Remote']
plot_data = master_vis[master_vis['remoteness_label'].notna()]
plot_data.boxplot(column='total_per_1000', by='remoteness_label',
                  ax=axes[1], positions=range(len(remote_order)),
                  notch=False, patch_artist=True)
axes[1].set_xticklabels(remote_order, rotation=20, ha='right')
axes[1].set_xlabel('')
axes[1].set_ylabel('Total Recipients per 1,000 Pop 70+')
axes[1].set_title('Utilisation Rate by Remoteness', fontweight='bold')
plt.suptitle('')

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_04_utilisation_rates.png', dpi=150)
plt.show()

---
## 5. Disadvantage vs Service Utilisation

Testing the relationship between IRSD (disadvantage) and aged care utilisation by remoteness.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Filter to SA2s with SEIFA data and non-zero utilisation
plot_df = master[master['irsd_score'].notna() & (master['total_per_1000'] > 0)].copy()
cap99 = plot_df['total_per_1000'].quantile(0.99)
plot_df = plot_df[plot_df['total_per_1000'] <= cap99]

# IRSD vs total utilisation
sc = axes[0].scatter(plot_df['irsd_score'], plot_df['total_per_1000'],
                     c=plot_df['remoteness_cat'], cmap='RdYlGn_r',
                     alpha=0.6, s=40, edgecolors='white', linewidth=0.3,
                     vmin=1, vmax=5)
plt.colorbar(sc, ax=axes[0], label='Remoteness (1=City, 5=Very Remote)')
# Trend line
z = np.polyfit(plot_df['irsd_score'].dropna(), 
               plot_df.loc[plot_df['irsd_score'].notna(), 'total_per_1000'], 1)
p = np.poly1d(z)
x_line = np.linspace(plot_df['irsd_score'].min(), plot_df['irsd_score'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', alpha=0.8, label='Trend')
axes[0].set_xlabel('IRSD Score (lower = more disadvantaged)')
axes[0].set_ylabel('Total Recipients per 1,000 Pop 70+')
axes[0].set_title('Disadvantage vs Aged Care Utilisation', fontweight='bold')
axes[0].legend(fontsize=9)

# Growth rate by remoteness
growth_by_remote = (
    master[master['remoteness_label'].notna()]
    .groupby('remoteness_label')['growth_rate_70plus_2031']
    .mean()
    .reindex(['Major City', 'Inner Regional', 'Outer Regional', 'Remote', 'Very Remote'])
    * 100
)
growth_by_remote.plot(kind='barh', ax=axes[1], color='#4C72B0', edgecolor='white')
axes[1].set_xlabel('Mean 70+ Population Growth Rate 2024→2031 (%)')
axes[1].set_title('Projected Growth Rate by Remoteness', fontweight='bold')
for i, val in enumerate(growth_by_remote):
    axes[1].text(val + 0.2, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_05_irsd_remoteness.png', dpi=150)
plt.show()

---
## 6. Correlation Heatmap — Key Features

In [ ]:
corr_cols = [
    'pop_70plus', 'pct_70plus_2024', 'growth_rate_70plus_2031',
    'total_per_1000', 'residential_per_1000', 'home_care_per_1000', 'chsp_per_1000',
    'irsd_score', 'irsad_score', 'remoteness_cat',
]
# Only use rows with complete data
corr_df = master[corr_cols].dropna()
corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Aged Care Demand Features', fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'eda_06_correlation.png', dpi=150)
plt.show()
print(f'Correlation computed on {len(corr_df):,} SA2 regions with complete data')

---
## 7. Top 20 SA2 Regions by Demand Risk Indicators

In [ ]:
# Simple demand pressure score:
# high elderly share + high projected growth + high utilisation rate
master['demand_pressure'] = (
    master['pct_70plus_2024'].fillna(0) * 0.4 +
    master['growth_rate_70plus_2031'].fillna(0) * 0.3 +
    (master['total_per_1000'].clip(0, master['total_per_1000'].quantile(0.95)) /
     master['total_per_1000'].quantile(0.95)) * 0.3
)

top20 = (
    master[master['irsd_score'].notna()]
    .nlargest(20, 'demand_pressure')
    [['sa2_code', 'sa2_name', 'state', 'pop_70plus', 'pct_70plus_2024',
      'growth_rate_70plus_2031', 'total_per_1000', 'remoteness_label',
      'irsd_score', 'demand_pressure']]
    .reset_index(drop=True)
)

print('Top 20 SA2 regions by demand pressure score:')
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 200)
display(top20)

---
## 8. EDA Summary

**Key findings:**

1. **Rapid ageing:** Australia's 70+ population is projected to grow ~28–35% by 2031 depending on state, with NT and QLD growing fastest.
2. **Utilisation patterns:** Residential care utilisation varies significantly by remoteness — remote areas show different patterns to major cities, reflecting both need and access barriers.
3. **Disadvantage correlation:** Lower IRSD scores (higher disadvantage) do not straightforwardly predict higher utilisation — remote disadvantaged areas may have *lower* utilisation due to access barriers, not lower need.
4. **Data limitation:** AIHW data is a single snapshot (2024/25), not a time series. Quarterly trend analysis (Q2 in problem framing) requires historical GEN data.
5. **SA2 coverage gaps:** ~100 SA2s in the hierarchy have no SEIFA data (unpopulated/suppressed) — handled via left join.

**Features ready for notebook 04:**
- `pop_70plus`, `pop_70plus_2031`, `pop_70plus_2041`, `pct_70plus_2024`, `growth_rate_70plus_2031`
- `total_per_1000`, `residential_per_1000`, `home_care_per_1000`, `chsp_per_1000`
- `irsd_score`, `irsad_score`, `remoteness_cat`
- `demand_pressure` (preliminary composite score)

**➡️ Next: [04_feature_engineering.ipynb](04_feature_engineering.ipynb)**